# **📦 1. Install Dependencies**

In [1]:
!pip install simpy


# **⚙️ 2. Import Libraries**

In [2]:
import simpy
import random
import numpy as np
import time
import pandas as pd
import itertools
import json
import sys


# **🧩 3. Simulation Configuration**

In [3]:
# Configuration
NUM_HOSTS = 5
HOST_CPU = 16
HOST_RAM = 32
VM_CPU_RANGE = (1, 4)
VM_RAM_RANGE = (2, 8)
VM_ARRIVAL_RATE = 0.8
AVG_VM_DURATION = 40.0
SLA_CPU_THRESHOLD = 0.9
PLACEMENT_LATENCY_RANGE = (10, 50)
POLICY = "round_robin"
SIM_DURATION = 100
SEED = 42

print(f"Using random seed: {SEED}")
random.seed(SEED)
np.random.seed(SEED)


Using random seed: 42


# **💾 4. Save Configuration to File**

In [4]:
# Save configuration
config = {
    "NUM_HOSTS": NUM_HOSTS,
    "HOST_CPU": HOST_CPU,
    "HOST_RAM": HOST_RAM,
    "VM_CPU_RANGE": VM_CPU_RANGE,
    "VM_RAM_RANGE": VM_RAM_RANGE,
    "VM_ARRIVAL_RATE": VM_ARRIVAL_RATE,
    "AVG_VM_DURATION": AVG_VM_DURATION,
    "SLA_CPU_THRESHOLD": SLA_CPU_THRESHOLD,
    "PLACEMENT_LATENCY_RANGE": PLACEMENT_LATENCY_RANGE,
    "POLICY": POLICY,
    "SIM_DURATION": SIM_DURATION,
    "SEED": SEED
}

with open("simulation_config.json", "w") as f:
    json.dump(config, f, indent=4)
print("Saved configuration to 'simulation_config.json'")


Saved configuration to 'simulation_config.json'


# **🏗️ 5. Define Host Class**

In [5]:
class Host:
    def __init__(self, env, id, total_cpu, total_ram):
        self.env = env
        self.id = id
        self.total_cpu = total_cpu
        self.total_ram = total_ram
        self.available_cpu = total_cpu
        self.available_ram = total_ram
        self.vms = []
        print(f"Host {self.id} created. CPU: {self.total_cpu}, RAM: {self.total_ram}")

    def allocate(self, vm):
        if self.available_cpu >= vm.cpu and self.available_ram >= vm.ram:
            self.available_cpu -= vm.cpu
            self.available_ram -= vm.ram
            self.vms.append(vm)
            vm.host = self
            print(f"{self.env.now:.2f}: Host {self.id}: Allocated VM {vm.id} ({vm.cpu} CPU, {vm.ram} RAM). "
                  f"Usage: {self.used_cpu()}/{self.total_cpu} CPU, {self.used_ram()}/{self.total_ram} RAM")
            return True
        return False

    def deallocate(self, vm):
        if vm in self.vms:
            self.available_cpu += vm.cpu
            self.available_ram += vm.ram
            self.vms.remove(vm)
            print(f"{self.env.now:.2f}: Host {self.id}: Deallocated VM {vm.id} ({vm.cpu} CPU, {vm.ram} RAM). "
                  f"Usage: {self.used_cpu()}/{self.total_cpu} CPU, {self.used_ram()}/{self.total_ram} RAM")
        else:
            print(f"Warning: Attempted to deallocate VM {vm.id} not found on Host {self.id}.")

    def get_cpu_utilization(self):
        return self.used_cpu() / self.total_cpu if self.total_cpu > 0 else 0

    def get_ram_utilization(self):
        return self.used_ram() / self.total_ram if self.total_ram > 0 else 0

    def used_cpu(self):
        return self.total_cpu - self.available_cpu

    def used_ram(self):
        return self.total_ram - self.available_ram

    def __repr__(self):
        return f"Host(id={self.id}, CPU={self.used_cpu()}/{self.total_cpu}, RAM={self.used_ram()}/{self.total_ram})"


# **💻 6. Define VM Class**

In [6]:
class VM:
    _ids = itertools.count(0)

    def __init__(self, env, cpu, ram, duration):
        self.env = env
        self.id = next(VM._ids)
        self.cpu = cpu
        self.ram = ram
        self.duration = duration
        self.host = None
        self.arrival_time = env.now
        self.departure_time = env.now + duration

    def __repr__(self):
        return f"VM(id={self.id}, CPU={self.cpu}, RAM={self.ram}, duration={self.duration:.2f})"


# **📊 7. Metrics Initialization and Host Iterator**

In [7]:
metrics = []
host_iterator = None


# **🔄 8. Round Robin Host Selection**

In [8]:
def get_next_host_round_robin(hosts):
    global host_iterator
    if host_iterator is None:
        host_iterator = itertools.cycle(hosts)
    return next(host_iterator)


# **🔁 9. VM Lifecycle Process**

In [9]:
def vm_lifecycle(env, vm, hosts, policy="round_robin"):
    placement_success = False
    chosen_host = None
    decision_start_time = time.perf_counter()

    if policy == "round_robin":
        attempts = 0
        initial_host_index = random.randint(0, len(hosts) - 1)
        for i in range(len(hosts)):
            host_index = (initial_host_index + i) % len(hosts)
            host = hosts[host_index]
            attempts += 1
            if host.allocate(vm):
                placement_success = True
                chosen_host = host
                break
    else:
        print(f"Warning: Unknown allocation policy '{policy}' specified.")
        placement_success = False

    decision_latency_ms = (time.perf_counter() - decision_start_time) * 1000
    placement_latency_ms = random.uniform(*PLACEMENT_LATENCY_RANGE) if placement_success else 0

    host_id = chosen_host.id if placement_success else -1
    cpu_util = chosen_host.get_cpu_utilization() if chosen_host else 0
    ram_util = chosen_host.get_ram_utilization() if chosen_host else 0
    sla_violated = cpu_util > SLA_CPU_THRESHOLD if chosen_host else False

    metrics.append({
        "time": env.now,
        "event_type": "placement" if placement_success else "rejection",
        "vm_id": vm.id,
        "required_cpu": vm.cpu,
        "required_ram": vm.ram,
        "host_id": host_id,
        "host_cpu_util_after": round(cpu_util, 3),
        "host_ram_util_after": round(ram_util, 3),
        "sla_violated_after": sla_violated,
        "placement_latency_ms": round(placement_latency_ms, 2),
        "decision_latency_ms": round(decision_latency_ms, 2),
        "allocation_success": int(placement_success),
        "policy": policy,
        "vm_duration": round(vm.duration, 2)
    })

    if placement_success:
        print(f"{env.now:.2f}: VM {vm.id}: Successfully placed on Host {host_id}. Running for {vm.duration:.2f} time units.")
        yield env.timeout(vm.duration)
        print(f"{env.now:.2f}: VM {vm.id}: Lifetime ended. Departing from Host {host_id}.")
        chosen_host.deallocate(vm)
    else:
        print(f"{env.now:.2f}: VM {vm.id}: Failed to place (Policy: {policy}). VM rejected.")


# **🧬 10. VM Generator Process**

In [10]:
def vm_generator(env, hosts, arrival_rate, avg_duration, policy):
    print(f"\n--- Starting VM Generator (Arrival Rate λ={arrival_rate}, Avg Duration μ={avg_duration}) ---")
    while True:
        interarrival_time = np.random.exponential(scale=1.0 / arrival_rate)
        yield env.timeout(interarrival_time)

        vm_cpu = random.randint(*VM_CPU_RANGE)
        vm_ram = random.randint(*VM_RAM_RANGE)
        vm_duration = max(0.1, np.random.exponential(scale=avg_duration))
        new_vm = VM(env, vm_cpu, vm_ram, vm_duration)

        print(f"\n{env.now:.2f}: VM_ARRIVAL - VM {new_vm.id} arrived requesting ({new_vm.cpu} CPU, {new_vm.ram} RAM) for {new_vm.duration:.2f} time units.")
        env.process(vm_lifecycle(env, new_vm, hosts, policy))


# **▶️ 11. Main Simulation Execution**

In [11]:
def main():
    print("--- Initializing Simulation ---")
    env = simpy.Environment()
    hosts = [Host(env, i, HOST_CPU, HOST_RAM) for i in range(NUM_HOSTS)]

    if not hosts:
        print("Error: Simulation requires at least one host.")
        return

    env.process(vm_generator(env, hosts, VM_ARRIVAL_RATE, AVG_VM_DURATION, POLICY))
    print(f"\n--- Running Simulation for {SIM_DURATION} time units ---")
    env.run(until=SIM_DURATION)
    print(f"\n--- Simulation Ended at Time: {env.now:.2f} ---")

    if not metrics:
        print("No metrics were recorded during the simulation.")
        return

    df = pd.DataFrame(metrics)
    df.to_csv("improved_cloud_allocation_metrics.csv", index=False)
    df.to_json("improved_cloud_allocation_metrics.json", orient="records", indent=2)
    print("Saved metrics to 'improved_cloud_allocation_metrics.csv' and '...json'")

    summary = {
        "total_vms": df['vm_id'].nunique(),
        "successful_placements": int((df['allocation_success'] == 1).sum()),
        "rejections": int((df['allocation_success'] == 0).sum()),
        "avg_decision_latency_ms": df[df['allocation_success'] == 1]['decision_latency_ms'].mean(),
        "avg_placement_latency_ms": df[df['allocation_success'] == 1]['placement_latency_ms'].mean(),
        "avg_cpu_util": df[df['allocation_success'] == 1]['host_cpu_util_after'].mean(),
        "avg_ram_util": df[df['allocation_success'] == 1]['host_ram_util_after'].mean(),
        "total_sla_violations": int(df[df['allocation_success'] == 1]['sla_violated_after'].sum())
    }
    with open("summary_statistics.json", "w") as f:
        json.dump(summary, f, indent=4)
    print("Saved summary statistics to 'summary_statistics.json'")

    vm_host_mapping = df[df['event_type'] == "placement"][["vm_id", "host_id"]]
    vm_host_mapping.to_csv("vm_host_mapping.csv", index=False)
    print("Saved VM-Host mapping to 'vm_host_mapping.csv'")

    print("\n--- First 5 Metric Records ---")
    print(df.head())


# **🧾 12. Run Simulation & Redirect Logs**

In [12]:
# Optional: Save logs to file
log_file = open("simulation_log.txt", "w")
sys.stdout = log_file
main()
sys.stdout = sys.__stdout__
log_file.close()
print("Saved full simulation log to 'simulation_log.txt'")
